# Week 4: Building Production Clinical Systems
**FURP 2026 — Cancer Detection Using Artificial Intelligence**  
**Instructor: Prof. Elio Espejo**

---

## 本周目标
1. 临床级 AI 系统：输入验证 / 审计日志 / 错误处理
2. 临床报告生成（标准化术语）
3. AI Assistant Tasks：DICOM处理 / 性能监控 / 法规文档 / REST API 设计

## 从研究到临床的核心差距
| 维度 | 研究原型 | 临床系统 |
|------|---------|----------|
| 输入 | 干净的标准图像 | 任意质量的DICOM文件 |
| 错误处理 | 允许崩溃 | 必须优雅降级 |
| 日志 | 无要求 | 监管合规强制要求 |
| 报告 | 数字输出 | 标准化医学术语 |
| 验证 | 学术数据集 | 多中心临床试验 |

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
import logging
import json
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime
from sklearn.metrics import (accuracy_score, recall_score, roc_auc_score,
                             confusion_matrix, roc_curve)
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备：{device}')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# 复用核心组件
def create_synthetic_mammograms(num_samples=500):
    np.random.seed(42)
    images, labels = [], []
    image_size = 256
    y, x = np.ogrid[:image_size, :image_size]
    for i in range(num_samples):
        image = np.random.normal(120, 25, (image_size, image_size))
        image += 20 * np.sin(x/30) * np.cos(y/25)
        has_cancer = np.random.random() < 0.3
        if has_cancer:
            cx = np.random.randint(40, image_size-40)
            cy = np.random.randint(40, image_size-40)
            r  = np.random.randint(15, 25)
            for dy in range(-r, r):
                for dx in range(-r, r):
                    if dx**2+dy**2 < r**2 and 0<=cy+dy<image_size and 0<=cx+dx<image_size:
                        image[cy+dy, cx+dx] += 60 + np.random.normal(0, 5)
            labels.append(1)
        else:
            labels.append(0)
        image = np.clip(image + np.random.normal(0,10,image.shape), 0, 255).astype(np.uint8)
        images.append(image)
    return np.array(images), np.array(labels)

class MedicalCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(True), nn.AdaptiveAvgPool2d((1,1))
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256,128), nn.ReLU(True), nn.Dropout(0.3), nn.Linear(128,num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0),-1))

images, labels = create_synthetic_mammograms(500)
model = MedicalCNN().to(device)
model.eval()
print(f'系统初始化完成 | 数据：{len(images)}张')

---
## Step 1: 临床级 AI 系统

**核心设计原则：**
1. 输入验证：拒绝不合格图像，而非产生错误结果
2. 审计日志：每次分析都有完整记录（监管要求）
3. 不确定性量化：高不确定性自动转人工
4. 优雅降级：任何错误不能中断临床流程

In [ ]:
class ClinicalMedicalAI:
    """
    生产级医学 AI 系统
    包含：输入验证 / 审计日志 / 临床报告 / 错误处理 / 不确定性量化
    """

    def __init__(self, model, device='cpu'):
        self.model   = model
        self.device  = device
        self.version = '1.0.0'
        self.fda_id  = 'K123456789'
        self.calibration_date = '2025-01-15'
        # 决策阈值（通过验证研究设定）
        self.cancer_threshold       = 0.5
        self.high_uncertainty_thr   = 0.3
        self.low_confidence_thr     = 0.2
        # 图像质量参数
        self.min_size     = (224, 224)
        self.max_size     = (1024, 1024)
        self.min_contrast = 15
        self.analysis_log = []
        self._setup_logger()

    def _setup_logger(self):
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s | %(levelname)s | %(message)s'
        )
        self.logger = logging.getLogger('ClinicalAI')
        self.logger.info(f'系统初始化 v{self.version} | FDA:{self.fda_id}')

    def validate_input(self, image):
        """严格的输入验证"""
        issues = []
        if not isinstance(image, np.ndarray):         issues.append('输入必须是numpy数组')
        elif len(image.shape) != 2:                   issues.append('必须是2D灰度图像')
        else:
            h, w = image.shape
            if h<self.min_size[0] or w<self.min_size[1]: issues.append(f'图像过小：{image.shape}')
            if h>self.max_size[0] or w>self.max_size[1]: issues.append(f'图像过大：{image.shape}')
            if np.isnan(image).any():                     issues.append('包含NaN值')
            if np.isinf(image).any():                     issues.append('包含Inf值')
            if np.std(image) < self.min_contrast:         issues.append(f'对比度不足：{np.std(image):.1f}')
        quality_score = min(1.0, np.std(image)/50.0) if not issues else 0.0
        return len(issues)==0, issues, quality_score

    def preprocess(self, image):
        """临床标准预处理"""
        resized = cv2.resize(image, (224,224), interpolation=cv2.INTER_CUBIC)
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5],[0.5])
        ])
        return transform(Image.fromarray(resized, mode='L')).unsqueeze(0).to(self.device)

    def predict_with_uncertainty(self, tensor, n_samples=50):
        """Monte Carlo Dropout 不确定性估计"""
        def enable_dropout(m):
            if isinstance(m, nn.Dropout): m.train()
        self.model.eval(); self.model.apply(enable_dropout)
        preds = []
        with torch.no_grad():
            for _ in range(n_samples):
                preds.append(F.softmax(self.model(tensor), dim=1).cpu().numpy())
        preds = np.array(preds)
        return float(np.mean(preds[:,0,1])), float(np.std(preds[:,0,1]))

    def analyze(self, image, patient_id=None, study_id=None, context=None):
        """完整临床分析流程"""
        analysis_id = f'ANALYSIS_{datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:20]}'
        self.logger.info(f'分析请求 | ID:{analysis_id} | 患者:{patient_id}')

        try:
            # Step 1: 验证
            valid, issues, quality = self.validate_input(image)
            if not valid:
                self.logger.error(f'验证失败 | {issues}')
                return {'status':'VALIDATION_FAILED','issues':issues,'analysis_id':analysis_id}

            # Step 2: 预处理
            tensor = self.preprocess(image)

            # Step 3: AI 分析
            cancer_prob, uncertainty = self.predict_with_uncertainty(tensor)

            # Step 4: 临床决策逻辑
            if uncertainty > self.high_uncertainty_thr:
                recommendation = 'HIGH_UNCERTAINTY'
                priority = 'HIGH'
                timeline = '24小时内'
            elif cancer_prob > 0.7:
                recommendation = 'SUSPICIOUS'
                priority = 'URGENT'
                timeline = '1-3天内'
            elif cancer_prob > 0.3:
                recommendation = 'BORDERLINE'
                priority = 'MODERATE'
                timeline = '1-2周内'
            else:
                recommendation = 'NORMAL'
                priority = 'ROUTINE'
                timeline = '下次常规筛查'

            # Step 5: 生成报告
            report = {
                'report_header': {
                    'analysis_id':    analysis_id,
                    'timestamp':      datetime.now().isoformat(),
                    'patient_id':     patient_id,
                    'study_id':       study_id,
                    'system_version': self.version,
                    'fda_clearance':  self.fda_id
                },
                'image_quality': {
                    'score':  round(quality, 3),
                    'rating': 'Excellent' if quality>0.8 else 'Good' if quality>0.6 else 'Adequate'
                },
                'ai_results': {
                    'cancer_probability':   round(cancer_prob, 3),
                    'uncertainty_estimate': round(uncertainty, 3),
                    'primary_finding':      recommendation,
                    'requires_expert':      uncertainty > self.high_uncertainty_thr
                },
                'clinical_workflow': {
                    'priority': priority,
                    'timeline': timeline,
                    'action':   f'{recommendation} - 请参考临床指南'
                },
                'limitations': [
                    '本系统仅作决策支持，最终诊断由执业放射科医生作出',
                    f'系统最后校准日期：{self.calibration_date}',
                    '系统性能在不同患者群体间可能存在差异'
                ]
            }

            # Step 6: 记录审计日志
            log_entry = {
                'analysis_id': analysis_id,
                'patient_id':  patient_id,
                'timestamp':   datetime.now().isoformat(),
                'finding':     recommendation,
                'probability': round(cancer_prob, 3),
                'uncertainty': round(uncertainty, 3)
            }
            self.analysis_log.append(log_entry)
            self.logger.info(f'分析完成 | {analysis_id} | {recommendation} | P={cancer_prob:.3f}')

            return {'status':'SUCCESS', 'report':report}

        except Exception as e:
            self.logger.error(f'系统错误 | {analysis_id} | {str(e)}')
            return {
                'status': 'SYSTEM_ERROR',
                'error':  '内部系统错误，请联系技术支持',
                'analysis_id': analysis_id
            }


# 初始化并测试
clinical_ai = ClinicalMedicalAI(model, device=str(device))

print('\n=== 临床分析演示 ===')
sample = images[np.where(labels==1)[0][0]]
result = clinical_ai.analyze(
    sample,
    patient_id='PATIENT_001',
    study_id='MAMMO_20260101_001',
    context={'age': 52, 'family_history': True}
)

if result['status'] == 'SUCCESS':
    r = result['report']
    print(f"分析ID：   {r['report_header']['analysis_id']}")
    print(f"图像质量： {r['image_quality']['rating']} ({r['image_quality']['score']})")
    print(f"癌症概率： {r['ai_results']['cancer_probability']}")
    print(f"不确定性： {r['ai_results']['uncertainty_estimate']}")
    print(f"主要发现： {r['ai_results']['primary_finding']}")
    print(f"临床优先级：{r['clinical_workflow']['priority']}")
    print(f"时限：    {r['clinical_workflow']['timeline']}")

---
## AI Assistant Task 1: DICOM 处理

**DICOM** = Digital Imaging and Communications in Medicine  
医院里所有医学图像的标准格式，包含图像数据 + 患者元数据

In [ ]:
class DICOMProcessor:
    """
    DICOM 图像处理器（模拟真实DICOM工作流）
    真实环境中使用 pydicom 库读取 .dcm 文件
    """

    def __init__(self):
        self.supported_modalities = ['MG', 'CR', 'CT', 'MR']

    def simulate_dicom_metadata(self, patient_id, study_id):
        """模拟DICOM标签（真实场景来自.dcm文件头）"""
        return {
            'PatientID':          patient_id,
            'PatientAge':         f'{np.random.randint(35,75):03d}Y',
            'StudyDate':          datetime.now().strftime('%Y%m%d'),
            'Modality':           'MG',
            'Manufacturer':       'Hologic',
            'KVP':                '28',
            'ExposureTime':       '800',
            'PixelSpacing':       '0.1\\0.1',
            'BitsAllocated':      16,
            'RescaleSlope':       1.0,
            'RescaleIntercept':   0.0,
            'WindowCenter':       2000,
            'WindowWidth':        4000
        }

    def windowing(self, image, center, width):
        """
        DICOM 窗宽/窗位调整
        控制显示的亮度范围，突出特定组织的对比度
        low  = center - width/2
        high = center + width/2
        """
        low  = center - width / 2
        high = center + width / 2
        windowed = np.clip(image, low, high)
        return ((windowed - low) / (high - low) * 255).astype(np.uint8)

    def process_image(self, raw_image, metadata):
        """完整DICOM处理流程"""
        # 1. 应用RescaleSlope/Intercept（灰度值校正）
        slope     = metadata.get('RescaleSlope', 1.0)
        intercept = metadata.get('RescaleIntercept', 0.0)
        hounsfield = raw_image * slope + intercept
        # 2. 窗宽窗位
        windowed = self.windowing(hounsfield,
                                   metadata['WindowCenter'],
                                   metadata['WindowWidth'])
        # 3. CLAHE 对比度增强
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(windowed)
        return enhanced

    def validate_modality(self, metadata):
        """验证成像方式是否受支持"""
        return metadata.get('Modality') in self.supported_modalities


# 演示DICOM处理流程
processor = DICOMProcessor()
sample_raw = images[0].astype(np.float32) * 20  # 模拟16位DICOM值
meta = processor.simulate_dicom_metadata('P001', 'S001')
processed = processor.process_image(sample_raw, meta)

# 可视化不同窗宽窗位的效果
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('DICOM Windowing: Different Window Settings', fontsize=14, fontweight='bold')

settings = [
    ('原始图像', images[0], None),
    ('软组织窗\n(C=400,W=1000)', None, (400, 1000)),
    ('乳腺窗\n(C=2000,W=4000)', None, (2000, 4000)),
    ('CLAHE增强', processed, None)
]

for ax, (title, img, window) in zip(axes, settings):
    if img is not None:
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    else:
        windowed = processor.windowing(sample_raw, *window)
        ax.imshow(windowed, cmap='gray')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('task1_dicom_processing.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nDICOM 元数据示例：')
for k, v in meta.items():
    print(f'  {k:<25}: {v}')
print('✅ Task 1 完成！')

---
## AI Assistant Task 2: 性能监控仪表盘

临床部署后需要持续监控：模型漂移、性能退化、设备差异

In [ ]:
class PerformanceMonitor:
    """
    生产环境性能监控系统
    检测：数据漂移 / 性能退化 / 系统异常
    """

    def __init__(self):
        self.metrics_history = []
        self.alert_thresholds = {
            'sensitivity': 0.85,  # 敏感性低于此值触发警报
            'specificity': 0.70,
            'auc':         0.90,
            'latency_ms':  500    # 推断延迟超过500ms触发警报
        }

    def record_batch(self, y_true, y_pred, y_proba, timestamp=None):
        if timestamp is None:
            timestamp = datetime.now()
        sens = recall_score(y_true, y_pred, zero_division=0)
        tn   = np.sum((y_pred==0)&(y_true==0))
        fp   = np.sum((y_pred==1)&(y_true==0))
        spec = tn/(tn+fp) if (tn+fp)>0 else 0
        auc  = roc_auc_score(y_true, y_proba) if len(np.unique(y_true))>1 else 0
        entry = {
            'timestamp':   timestamp,
            'sensitivity': sens,
            'specificity': spec,
            'auc':         auc,
            'n_samples':   len(y_true),
            'cancer_rate': np.mean(y_true)
        }
        self.metrics_history.append(entry)
        return entry

    def check_alerts(self, entry):
        alerts = []
        if entry['sensitivity'] < self.alert_thresholds['sensitivity']:
            alerts.append(f'⚠️  敏感性下降：{entry["sensitivity"]:.3f} < {self.alert_thresholds["sensitivity"]}')
        if entry['auc'] < self.alert_thresholds['auc']:
            alerts.append(f'⚠️  AUC下降：{entry["auc"]:.3f} < {self.alert_thresholds["auc"]}')
        return alerts

    def drift_detection(self):
        """简单漂移检测：比较近期与历史性能"""
        if len(self.metrics_history) < 4:
            return None
        recent = self.metrics_history[-2:]
        hist   = self.metrics_history[:-2]
        recent_auc = np.mean([m['auc'] for m in recent])
        hist_auc   = np.mean([m['auc'] for m in hist])
        drift = hist_auc - recent_auc
        return {'drift': drift, 'significant': drift > 0.05}


# 模拟多批次监控数据
monitor = PerformanceMonitor()
X_tr, X_te, y_tr, y_te = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels)

transform = transforms.Compose([transforms.Resize((64,64)), transforms.ToTensor(), transforms.Normalize([0.5],[0.5])])
val_ds = type('D', (object,), {
    '__len__': lambda s: len(X_te),
    '__getitem__': lambda s,i: (
        transform(Image.fromarray(X_te[i], mode='L')),
        torch.tensor(y_te[i], dtype=torch.long)
    )
})()

print('模拟6个月生产监控数据...')
import time
from datetime import timedelta

base_date = datetime(2026, 1, 1)
for month in range(6):
    # 模拟不同月份：后期引入轻微性能退化
    noise = np.random.normal(0, 0.02 + month*0.01)
    y_pred_m = (np.random.random(len(y_te)) > (0.5 + noise)).astype(int)
    y_prob_m = np.random.beta(2+month*0.1, 3) * np.ones(len(y_te))
    # 部分保留真实信号
    mask = np.random.random(len(y_te)) > 0.3
    y_pred_m[mask] = y_te[mask]
    y_prob_m[y_te==1] = 0.7 - month*0.03
    y_prob_m[y_te==0] = 0.2 + month*0.01
    y_prob_m = np.clip(y_prob_m, 0, 1)

    ts = base_date + timedelta(days=30*month)
    entry = monitor.record_batch(y_te, y_pred_m, y_prob_m, timestamp=ts)
    alerts = monitor.check_alerts(entry)
    print(f'月份{month+1} | Sens:{entry["sensitivity"]:.3f} | Spec:{entry["specificity"]:.3f} | AUC:{entry["auc"]:.3f}', end='')
    if alerts:
        print(f' | {alerts[0]}')
    else:
        print(' | ✓ 正常')

drift_result = monitor.drift_detection()
if drift_result:
    print(f'\n漂移检测：AUC下降={drift_result["drift"]:.3f} | 显著={drift_result["significant"]}')

# 可视化监控仪表盘
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Clinical AI Performance Monitoring Dashboard', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

months = [m['timestamp'].strftime('%m/%Y') for m in monitor.metrics_history]
sens_vals = [m['sensitivity'] for m in monitor.metrics_history]
spec_vals = [m['specificity'] for m in monitor.metrics_history]
auc_vals  = [m['auc']         for m in monitor.metrics_history]

# 敏感性趋势
ax1 = fig.add_subplot(gs[0,0])
ax1.plot(months, sens_vals, 'o-', color='#E24B4A', lw=2, markersize=8, label='Sensitivity')
ax1.axhline(y=0.85, color='#E24B4A', linestyle='--', lw=1.5, alpha=0.7, label='Alert threshold')
ax1.fill_between(months, [0.85]*len(months), sens_vals,
                 where=[s<0.85 for s in sens_vals], alpha=0.3, color='#E24B4A')
ax1.set_title('Sensitivity Trend', fontweight='bold')
ax1.set_ylim([0.5, 1.05]); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=30)

# AUC 趋势
ax2 = fig.add_subplot(gs[0,1])
ax2.plot(months, auc_vals, 's-', color='#185FA5', lw=2, markersize=8, label='AUC-ROC')
ax2.axhline(y=0.90, color='#185FA5', linestyle='--', lw=1.5, alpha=0.7, label='Alert threshold')
ax2.set_title('AUC-ROC Trend', fontweight='bold')
ax2.set_ylim([0.5, 1.05]); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=30)

# 综合雷达图（最新月份）
ax3 = fig.add_subplot(gs[0,2], projection='polar')
cats = ['Sensitivity', 'Specificity', 'AUC']
vals = [sens_vals[-1], spec_vals[-1], auc_vals[-1]]
angles = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist()
vals  += [vals[0]]; angles += [angles[0]]
ax3.plot(angles, vals, 'o-', color='#185FA5', lw=2)
ax3.fill(angles, vals, alpha=0.2, color='#185FA5')
thrs = [0.85, 0.70, 0.90]; thrs += [thrs[0]]
ax3.plot(angles, thrs, '--', color='#E24B4A', lw=1.5, alpha=0.7)
ax3.set_xticks(angles[:-1]); ax3.set_xticklabels(cats, fontsize=10)
ax3.set_ylim([0, 1.0]); ax3.set_title('Latest Performance\nRadar', fontweight='bold')

# 月度对比条形
ax4 = fig.add_subplot(gs[1,0:2])
x = np.arange(len(months)); w = 0.25
ax4.bar(x-w, sens_vals, w, label='Sensitivity', color='#E24B4A', alpha=0.8)
ax4.bar(x,   spec_vals, w, label='Specificity', color='#185FA5', alpha=0.8)
ax4.bar(x+w, auc_vals,  w, label='AUC',         color='#1D9E75', alpha=0.8)
ax4.set_xticks(x); ax4.set_xticklabels(months, rotation=30, fontsize=9)
ax4.set_ylabel('Score'); ax4.set_ylim([0, 1.1])
ax4.set_title('Monthly Performance Comparison', fontweight='bold')
ax4.legend(fontsize=10); ax4.grid(True, alpha=0.3, axis='y')

# 漂移检测
ax5 = fig.add_subplot(gs[1,2])
ax5.text(0.5, 0.7, '性能漂移分析', ha='center', va='center',
         fontsize=14, fontweight='bold', transform=ax5.transAxes)
drift_val = drift_result['drift'] if drift_result else 0
color = '#E24B4A' if drift_result and drift_result['significant'] else '#1D9E75'
ax5.text(0.5, 0.5, f'AUC 下降：{drift_val:.3f}', ha='center', va='center',
         fontsize=13, color=color, transform=ax5.transAxes)
status = '需要重新校准！' if drift_result and drift_result['significant'] else '性能稳定'
ax5.text(0.5, 0.3, status, ha='center', va='center',
         fontsize=12, color=color, fontweight='bold', transform=ax5.transAxes)
ax5.set_title('Drift Detection', fontweight='bold')
ax5.axis('off')

plt.savefig('task2_monitoring_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 2 完成！')

---
## AI Assistant Task 3: FDA 法规文档生成

In [ ]:
def generate_regulatory_document(model, images, labels):
    """生成 FDA 510(k) 申请所需的性能验证文档"""

    # 基础性能评估
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.ensemble import RandomForestClassifier

    X_tr, X_te, y_tr, y_te = train_test_split(
        images, labels, test_size=0.2, random_state=42, stratify=labels)

    transform = transforms.Compose([
        transforms.Resize((32,32)), transforms.ToTensor(), transforms.Normalize([0.5],[0.5])
    ])

    # 用简单特征模拟性能指标（避免完整CNN训练时间）
    from skimage.feature import local_binary_pattern
    def simple_feat(img):
        return [np.mean(img), np.std(img), np.mean(img>140)]

    Xf_tr = np.array([simple_feat(img) for img in X_tr])
    Xf_te = np.array([simple_feat(img) for img in X_te])
    sc = StandardScaler()
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf.fit(sc.fit_transform(Xf_tr), y_tr)
    y_pred  = rf.predict(sc.transform(Xf_te))
    y_proba = rf.predict_proba(sc.transform(Xf_te))[:,1]

    sens = recall_score(y_te, y_pred, zero_division=0)
    tn   = np.sum((y_pred==0)&(y_te==0)); fp = np.sum((y_pred==1)&(y_te==0))
    spec = tn/(tn+fp) if (tn+fp)>0 else 0
    auc_val = roc_auc_score(y_te, y_proba)

    doc = {
        'document_type': 'FDA 510(k) Premarket Notification',
        'device_name':   'MedicalAI Cancer Detection System v1.0',
        'submission_date': datetime.now().strftime('%Y-%m-%d'),
        'section_1_device_description': {
            'intended_use': '辅助放射科医生检测乳腺X光片中的可疑病变',
            'device_type':  '计算机辅助检测（CAD）系统',
            'technology':   '基于卷积神经网络的深度学习',
            'contraindications': '不适用于儿童或植入假体的患者'
        },
        'section_2_performance': {
            'primary_endpoint': f'AUC-ROC: {auc_val:.3f} (目标 ≥ 0.90)',
            'sensitivity':      f'{sens:.3f} (目标 ≥ 0.85)',
            'specificity':      f'{spec:.3f} (目标 ≥ 0.70)',
            'test_set_size':    len(y_te),
            'cancer_prevalence':f'{np.mean(y_te):.1%}'
        },
        'section_3_risk_analysis': {
            'primary_risk':       'False Negative（漏诊）',
            'risk_mitigation':    '系统仅作辅助工具，最终诊断由医生作出',
            'residual_risk':      '可接受（ALARP原则）',
            'risk_class':         'Class II 医疗器械'
        },
        'section_4_clinical_validation': {
            'study_design':   '回顾性多中心研究',
            'sample_size':    len(images),
            'data_source':    '3个三级医疗中心',
            'reader_study':   '与6名放射科医生对比'
        }
    }
    return doc


doc = generate_regulatory_document(model, images, labels)

print('='*60)
print('FDA 510(k) 性能验证文档摘要')
print('='*60)
for section, content in doc.items():
    if isinstance(content, dict):
        print(f'\n[{section}]')
        for k, v in content.items():
            print(f'  {k:<30}: {v}')
    else:
        print(f'{section}: {content}')

# 保存为JSON
with open('regulatory_document.json', 'w', encoding='utf-8') as f:
    json.dump(doc, f, ensure_ascii=False, indent=2)
print('\n✅ Task 3 完成！已保存：regulatory_document.json')

---
## AI Assistant Task 4: REST API 设计

临床集成需要 RESTful API：医院 HIS/PACS 系统通过 HTTP 调用 AI 分析

In [ ]:
# REST API 设计文档（实际部署使用 Flask/FastAPI）
api_spec = {
    'openapi': '3.0.0',
    'info': {
        'title':   'Medical AI Cancer Detection API',
        'version': '1.0.0',
        'description': 'REST API for clinical integration'
    },
    'endpoints': {
        'POST /api/v1/analyze': {
            'description': '提交图像进行癌症分析',
            'auth':        'Bearer Token (JWT)',
            'rate_limit':  '100 requests/hour',
            'request_body': {
                'image_base64': 'string (Base64编码的PNG/DICOM)',
                'patient_id':   'string',
                'study_id':     'string',
                'modality':     'string (MG/CT/MR)'
            },
            'response': {
                'analysis_id':       'string',
                'cancer_probability':'float [0,1]',
                'uncertainty':       'float [0,1]',
                'recommendation':    'string',
                'processing_ms':     'integer'
            },
            'status_codes': {'200':'成功', '400':'输入验证失败', '503':'系统过载'}
        },
        'GET /api/v1/health': {
            'description': '系统健康检查',
            'response': {'status':'string', 'model_version':'string', 'uptime_s':'integer'}
        },
        'GET /api/v1/metrics': {
            'description': '获取性能监控指标',
            'auth': 'Admin Token',
            'response': {'sensitivity':'float', 'specificity':'float', 'auc':'float', 'total_analyzed':'integer'}
        }
    }
}

# 模拟 Flask API 代码
flask_code = '''
from flask import Flask, request, jsonify
import base64
import numpy as np
from PIL import Image
import io
import time

app = Flask(__name__)
clinical_ai = ClinicalMedicalAI(model)  # 全局单例

@app.route('/api/v1/analyze', methods=['POST'])
def analyze():
    start_time = time.time()

    # 1. 认证
    token = request.headers.get('Authorization')
    if not validate_token(token):
        return jsonify({'error': 'Unauthorized'}), 401

    # 2. 解析请求
    data       = request.json
    img_bytes  = base64.b64decode(data['image_base64'])
    image      = np.array(Image.open(io.BytesIO(img_bytes)).convert('L'))

    # 3. AI 分析
    result = clinical_ai.analyze(
        image,
        patient_id=data.get('patient_id'),
        study_id=data.get('study_id')
    )

    # 4. 返回响应
    if result['status'] == 'SUCCESS':
        r = result['report']['ai_results']
        return jsonify({
            'analysis_id':        result['report']['report_header']['analysis_id'],
            'cancer_probability': r['cancer_probability'],
            'uncertainty':        r['uncertainty_estimate'],
            'recommendation':     r['primary_finding'],
            'processing_ms':      int((time.time()-start_time)*1000)
        }), 200
    else:
        return jsonify(result), 400

@app.route('/api/v1/health')
def health():
    return jsonify({'status': 'healthy', 'model_version': '1.0.0'})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, ssl_context='adhoc')  # HTTPS
'''

print('REST API 规范：')
print(json.dumps(api_spec, ensure_ascii=False, indent=2))

print('\n=== Flask API 实现代码 ===')
print(flask_code)

# 可视化 API 架构
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')
fig.suptitle('Clinical Integration Architecture', fontsize=15, fontweight='bold')

components = [
    (0.05, 0.5, 'HIS/PACS\n医院系统',    '#E6F1FB', '#185FA5'),
    (0.25, 0.5, 'API Gateway\n认证/限流', '#EAF3DE', '#3B6D11'),
    (0.50, 0.5, 'AI Service\nFlask API',  '#FAEEDA', '#854F0B'),
    (0.75, 0.5, 'CNN Model\nGPU推断',     '#FCEBEB', '#A32D2D'),
    (0.50, 0.1, '审计日志\n数据库',       '#EEEDFE', '#534AB7')
]

for x, y, text, fc, ec in components:
    ax.add_patch(plt.FancyBboxPatch((x-0.09, y-0.18), 0.18, 0.36,
                                    boxstyle='round,pad=0.02',
                                    facecolor=fc, edgecolor=ec, lw=2))
    ax.text(x, y, text, ha='center', va='center',
            fontsize=11, fontweight='bold', color=ec)

# 箭头
for x1, x2 in [(0.14,0.16), (0.34,0.41), (0.59,0.66)]:
    ax.annotate('', xy=(x2,0.5), xytext=(x1,0.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))
    ax.annotate('', xy=(x1,0.5), xytext=(x2,0.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2,
                                connectionstyle='arc3,rad=-0.2'))
ax.annotate('', xy=(0.5,0.28), xytext=(0.5,0.32),
            arrowprops=dict(arrowstyle='->', color='#534AB7', lw=2))

ax.text(0.5, 0.78, 'HTTPS / REST / JWT Auth', ha='center', fontsize=10,
        color='gray', style='italic')

plt.tight_layout()
plt.savefig('task4_api_architecture.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Task 4 完成！')

---
## 4周完整旅程总结

| 周次 | 核心技术 | 关键成果 |
|------|---------|----------|
| Week 1 | 传统CV + 随机森林 | 19个手工特征，AUC≈1.0（合成数据）|
| Week 2 | CNN + 数据增强 | 自动特征学习，端到端训练 |
| Week 3 | 不确定性量化 + 集成 | Grad-CAM可解释性，MC Dropout |
| Week 4 | 临床系统 + 法规 | 完整部署就绪系统 |

**核心洞见链：**
> 像素 → 手工特征 → 机器学习 → CNN自动学习 → 不确定性 → 临床部署

每一步都解决了上一步的局限性，这就是医学AI发展的缩影。

**你已经具备的能力：**
- ✅ 理解医学图像的数学本质
- ✅ 实现传统CV和深度学习的完整流程
- ✅ 处理类别不平衡和不确定性量化
- ✅ 构建临床级AI系统
- ✅ 理解FDA法规和临床集成

In [ ]:
print('='*60)
print('FURP 2026 — 4周医学AI研究项目完成！')
print('='*60)
print('所有输出文件：')
output_files = [
    'task1_dicom_processing.png',
    'task2_monitoring_dashboard.png',
    'regulatory_document.json',
    'task4_api_architecture.png',
]
for f in output_files:
    print(f'  {f}')
print('\n感谢 Prof. Elio Espejo 的指导！')